In [1]:
import os
import pandas as pd
import numpy as np

BASE_DIR   = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR   = BASE_DIR + 'processed_data/'
DB_DIR     = BASE_DIR + 'data_collection/databases_for_mapping/'

OUT_PATH   = BASE_DIR + 'processed_data_relation_wise_merge/generalised/PMID_CELLULARCOMPONENT/ALL_PMID_CELLULARCOMPONENT.parquet'

REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 2. Load KG Sources

#### CKG

In [2]:
CKG = pd.read_csv(PROC_DIR + 'CKG/File_13_PMID_Cellular_component_CKG.csv')
CKG


,Head,Relation,Tail,Head_type,Tail_type,KG_Source,Tail_GO_name,Head_ID_IS,Tail_ID_IS
0,PMID:20339535,MENTIONED_IN_PUBLICATION,GO:0000015,PMID,Cellular_component,CKG,phosphopyruvate hydratase complex,PubmedID,QuickGO
1,PMID:20067621,MENTIONED_IN_PUBLICATION,GO:0000015,PMID,Cellular_component,CKG,phosphopyruvate hydratase complex,PubmedID,QuickGO
2,PMID:14613499,MENTIONED_IN_PUBLICATION,GO:0000015,PMID,Cellular_component,CKG,phosphopyruvate hydratase complex,PubmedID,QuickGO
3,PMID:18523666,MENTIONED_IN_PUBLICATION,GO:0000015,PMID,Cellular_component,CKG,phosphopyruvate hydratase complex,PubmedID,QuickGO
4,PMID:19492051,MENTIONED_IN_PUBLICATION,GO:0000015,PMID,Cellular_component,CKG,phosphopyruvate hydratase complex,PubmedID,QuickGO
...,...,...,...,...,...,...,...,...,...
87097957,PMID:20724623,MENTIONED_IN_PUBLICATION,GO:1990973,PMID,Cellular_component,CKG,transmembrane actin-associated (TAN) line,PubmedID,QuickGO
87097958,PMID:15815710,MENTIONED_IN_PUBLICATION,GO:1990973,PMID,Cellular_component,CKG,transmembrane actin-associated (TAN) line,PubmedID,QuickGO
87097959,PMID:14719621,MENTIONED_IN_PUBLICATION,GO:1990973,PMID,Cellular_component,CKG,transmembrane actin-associated (TAN) line,PubmedID,QuickGO
87097960,PMID:11466488,MENTIONED_IN_PUBLICATION,GO:1990973,PMID,Cellular_component,CKG,transmembrane actin-associated (TAN) line,PubmedID,QuickGO


In [3]:
CKG.columns = CKG.columns.str.lower()
CKG.rename(columns={'tail_go_name': 'tail_detail_name'}, inplace=True)
CKG['relation'] = 'PMID_CellularComponent'
CKG['tail_type'] = 'CellularComponent'
CKG['kg_type'] = 'Generalised'
CKG['kg_source'] = 'CKG'
CKG['species'] = 'Homo species'
CKG

,head,relation,tail,head_type,tail_type,kg_source,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,PMID:20339535,PMID_CellularComponent,GO:0000015,PMID,CellularComponent,CKG,phosphopyruvate hydratase complex,PubmedID,QuickGO,Generalised,Homo species
1,PMID:20067621,PMID_CellularComponent,GO:0000015,PMID,CellularComponent,CKG,phosphopyruvate hydratase complex,PubmedID,QuickGO,Generalised,Homo species
2,PMID:14613499,PMID_CellularComponent,GO:0000015,PMID,CellularComponent,CKG,phosphopyruvate hydratase complex,PubmedID,QuickGO,Generalised,Homo species
3,PMID:18523666,PMID_CellularComponent,GO:0000015,PMID,CellularComponent,CKG,phosphopyruvate hydratase complex,PubmedID,QuickGO,Generalised,Homo species
4,PMID:19492051,PMID_CellularComponent,GO:0000015,PMID,CellularComponent,CKG,phosphopyruvate hydratase complex,PubmedID,QuickGO,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...
87097957,PMID:20724623,PMID_CellularComponent,GO:1990973,PMID,CellularComponent,CKG,transmembrane actin-associated (TAN) line,PubmedID,QuickGO,Generalised,Homo species
87097958,PMID:15815710,PMID_CellularComponent,GO:1990973,PMID,CellularComponent,CKG,transmembrane actin-associated (TAN) line,PubmedID,QuickGO,Generalised,Homo species
87097959,PMID:14719621,PMID_CellularComponent,GO:1990973,PMID,CellularComponent,CKG,transmembrane actin-associated (TAN) line,PubmedID,QuickGO,Generalised,Homo species
87097960,PMID:11466488,PMID_CellularComponent,GO:1990973,PMID,CellularComponent,CKG,transmembrane actin-associated (TAN) line,PubmedID,QuickGO,Generalised,Homo species


In [4]:
SOURCE_DFS = [
              ('CKG', CKG)
             ]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    print(f"[{name}] {'duplicate columns: ' + str(dupes) if dupes else '✓ no duplicates'}")

[CKG] ✓ no duplicates


In [5]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name','kg_type', 'species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.loc[:, ~df.columns.duplicated()].copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
print(f"Combined: {len(final_df):,} rows")
final_df

Combined: 87,097,962 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,PMID:20339535,PMID_CellularComponent,GO:0000015,PMID,<NA>,CellularComponent,CKG,PubmedID,QuickGO,<NA>,phosphopyruvate hydratase complex,Generalised,Homo species
1,PMID:20067621,PMID_CellularComponent,GO:0000015,PMID,<NA>,CellularComponent,CKG,PubmedID,QuickGO,<NA>,phosphopyruvate hydratase complex,Generalised,Homo species
2,PMID:14613499,PMID_CellularComponent,GO:0000015,PMID,<NA>,CellularComponent,CKG,PubmedID,QuickGO,<NA>,phosphopyruvate hydratase complex,Generalised,Homo species
3,PMID:18523666,PMID_CellularComponent,GO:0000015,PMID,<NA>,CellularComponent,CKG,PubmedID,QuickGO,<NA>,phosphopyruvate hydratase complex,Generalised,Homo species
4,PMID:19492051,PMID_CellularComponent,GO:0000015,PMID,<NA>,CellularComponent,CKG,PubmedID,QuickGO,<NA>,phosphopyruvate hydratase complex,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...
87097957,PMID:20724623,PMID_CellularComponent,GO:1990973,PMID,<NA>,CellularComponent,CKG,PubmedID,QuickGO,<NA>,transmembrane actin-associated (TAN) line,Generalised,Homo species
87097958,PMID:15815710,PMID_CellularComponent,GO:1990973,PMID,<NA>,CellularComponent,CKG,PubmedID,QuickGO,<NA>,transmembrane actin-associated (TAN) line,Generalised,Homo species
87097959,PMID:14719621,PMID_CellularComponent,GO:1990973,PMID,<NA>,CellularComponent,CKG,PubmedID,QuickGO,<NA>,transmembrane actin-associated (TAN) line,Generalised,Homo species
87097960,PMID:11466488,PMID_CellularComponent,GO:1990973,PMID,<NA>,CellularComponent,CKG,PubmedID,QuickGO,<NA>,transmembrane actin-associated (TAN) line,Generalised,Homo species


In [6]:
final_df[final_df['head_detail_name'].isna()]

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,PMID:20339535,PMID_CellularComponent,GO:0000015,PMID,<NA>,CellularComponent,CKG,PubmedID,QuickGO,<NA>,phosphopyruvate hydratase complex,Generalised,Homo species
1,PMID:20067621,PMID_CellularComponent,GO:0000015,PMID,<NA>,CellularComponent,CKG,PubmedID,QuickGO,<NA>,phosphopyruvate hydratase complex,Generalised,Homo species
2,PMID:14613499,PMID_CellularComponent,GO:0000015,PMID,<NA>,CellularComponent,CKG,PubmedID,QuickGO,<NA>,phosphopyruvate hydratase complex,Generalised,Homo species
3,PMID:18523666,PMID_CellularComponent,GO:0000015,PMID,<NA>,CellularComponent,CKG,PubmedID,QuickGO,<NA>,phosphopyruvate hydratase complex,Generalised,Homo species
4,PMID:19492051,PMID_CellularComponent,GO:0000015,PMID,<NA>,CellularComponent,CKG,PubmedID,QuickGO,<NA>,phosphopyruvate hydratase complex,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...
87097957,PMID:20724623,PMID_CellularComponent,GO:1990973,PMID,<NA>,CellularComponent,CKG,PubmedID,QuickGO,<NA>,transmembrane actin-associated (TAN) line,Generalised,Homo species
87097958,PMID:15815710,PMID_CellularComponent,GO:1990973,PMID,<NA>,CellularComponent,CKG,PubmedID,QuickGO,<NA>,transmembrane actin-associated (TAN) line,Generalised,Homo species
87097959,PMID:14719621,PMID_CellularComponent,GO:1990973,PMID,<NA>,CellularComponent,CKG,PubmedID,QuickGO,<NA>,transmembrane actin-associated (TAN) line,Generalised,Homo species
87097960,PMID:11466488,PMID_CellularComponent,GO:1990973,PMID,<NA>,CellularComponent,CKG,PubmedID,QuickGO,<NA>,transmembrane actin-associated (TAN) line,Generalised,Homo species


In [7]:
final_df[final_df['tail_detail_name'].isna()]

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species


In [8]:
import pandas as pd
import numpy as np

# Vectorized hash using numpy — avoids Python string ops entirely
keys = (
    pd.util.hash_array(final_df['head'].to_numpy()) ^
    pd.util.hash_array(final_df['relation'].to_numpy()) * 2654435761 ^
    pd.util.hash_array(final_df['tail'].to_numpy()) * 40503
)

# Keep first occurrence of each hash
mask = ~pd.Series(keys).duplicated(keep='first').to_numpy()
final_df = final_df.iloc[mask].reset_index(drop=True)

print(f"After dedup: {len(final_df):,} rows")
final_df

After dedup: 87,097,962 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,PMID:20339535,PMID_CellularComponent,GO:0000015,PMID,<NA>,CellularComponent,CKG,PubmedID,QuickGO,<NA>,phosphopyruvate hydratase complex,Generalised,Homo species
1,PMID:20067621,PMID_CellularComponent,GO:0000015,PMID,<NA>,CellularComponent,CKG,PubmedID,QuickGO,<NA>,phosphopyruvate hydratase complex,Generalised,Homo species
2,PMID:14613499,PMID_CellularComponent,GO:0000015,PMID,<NA>,CellularComponent,CKG,PubmedID,QuickGO,<NA>,phosphopyruvate hydratase complex,Generalised,Homo species
3,PMID:18523666,PMID_CellularComponent,GO:0000015,PMID,<NA>,CellularComponent,CKG,PubmedID,QuickGO,<NA>,phosphopyruvate hydratase complex,Generalised,Homo species
4,PMID:19492051,PMID_CellularComponent,GO:0000015,PMID,<NA>,CellularComponent,CKG,PubmedID,QuickGO,<NA>,phosphopyruvate hydratase complex,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...
87097957,PMID:20724623,PMID_CellularComponent,GO:1990973,PMID,<NA>,CellularComponent,CKG,PubmedID,QuickGO,<NA>,transmembrane actin-associated (TAN) line,Generalised,Homo species
87097958,PMID:15815710,PMID_CellularComponent,GO:1990973,PMID,<NA>,CellularComponent,CKG,PubmedID,QuickGO,<NA>,transmembrane actin-associated (TAN) line,Generalised,Homo species
87097959,PMID:14719621,PMID_CellularComponent,GO:1990973,PMID,<NA>,CellularComponent,CKG,PubmedID,QuickGO,<NA>,transmembrane actin-associated (TAN) line,Generalised,Homo species
87097960,PMID:11466488,PMID_CellularComponent,GO:1990973,PMID,<NA>,CellularComponent,CKG,PubmedID,QuickGO,<NA>,transmembrane actin-associated (TAN) line,Generalised,Homo species


In [9]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,87097962,0,87097962
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,87097962,0,87097962


In [10]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
final_df.to_parquet(OUT_PATH, index=False)
print(f"Saved {len(final_df):,} rows → {OUT_PATH}")

Saved 87,097,962 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/PMID_CELLULARCOMPONENT/ALL_PMID_CELLULARCOMPONENT.parquet


In [12]:
#